# G11project — Archived Directional Corridor v1 Retraining

This notebook preserves the rejected v1 diagnostic protocol. For the new single-incident experiment, use `colab_directional_corridor_v2.ipynb`.

Use **Runtime → Run all** with a GPU only when reproducing v1. This notebook installs SUMO, loads the directional-corridor training code, validates the scenario runtime, trains three multi-scenario candidates with validation-based selection, evaluates four methods on a locked test set, runs the directional behavior gate, and exports presentation figures.

Expected wall-clock budget: up to five hours. Results remain preliminary highway-only course-demo evidence.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Select a GPU runtime before using Runtime -> Run all"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

REPOSITORY = Path("/content/G11project").resolve()
REPOSITORY_URL = "https://github.com/yangyu-rgb/G11project.git"
PERSISTENT_ROOT = Path("/content/drive/MyDrive/G11project-directional-corridor-v1").resolve()
WORK_ROOT = Path("/content/g11project-directional-corridor-v1-work").resolve()
PIPELINE = REPOSITORY / "BackEnd/scripts/run_adaptive_demo_pipeline.py"
CONFIG = REPOSITORY / "BackEnd/configs/adaptive_demo_training_v1.yaml"
GENERATOR = REPOSITORY / "BackEnd/scripts/generate_highway_scenario.py"

if not (REPOSITORY / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    print("Updating repository already present in this runtime:", REPOSITORY)
    subprocess.run(["git", "-C", str(REPOSITORY), "pull", "--ff-only"], check=True)
os.chdir(REPOSITORY)
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

required_sources = [
    PIPELINE,
    REPOSITORY / "BackEnd/src/environment/adaptive_radius_wrapper.py",
    REPOSITORY / "BackEnd/src/environment/receiver_relevance.py",
    REPOSITORY / "BackEnd/src/experiments/presentation_gate.py",
]
missing_sources = [
    str(path.relative_to(REPOSITORY)) for path in required_sources if not path.is_file()
]
if missing_sources:
    revision = subprocess.run(
        ["git", "-C", str(REPOSITORY), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        check=False,
    ).stdout.strip()
    raise RuntimeError(
        "The GitHub checkout does not contain the directional-corridor training code. "
        f"Revision: {revision or 'unknown'}; missing: {', '.join(missing_sources)}. "
        "Commit and push the current project changes, delete /content/G11project, then rerun this notebook."
    )
print(
    "Training source revision:",
    revision
    if "revision" in globals()
    else subprocess.run(
        ["git", "-C", str(REPOSITORY), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip(),
)


subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "./BackEnd"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "eclipse-sumo==1.27.1"], check=True
)

import sumo  # noqa: E402

SUMO_HOME = Path(sumo.SUMO_HOME).resolve()
os.environ["SUMO_HOME"] = str(SUMO_HOME)
os.environ["PATH"] = str(SUMO_HOME / "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = str(SUMO_HOME / "tools") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("SUMO_HOME:", SUMO_HOME)
version = subprocess.run(["sumo", "--version"], capture_output=True, text=True, check=True)
print(version.stdout.splitlines()[0])

PREFLIGHT = Path("/content/g11-adaptive-demo-preflight")
preflight = subprocess.run(
    [sys.executable, "-B", str(GENERATOR), "--output", str(PREFLIGHT)],
    cwd=str(REPOSITORY),
    capture_output=True,
    text=True,
)
print(preflight.stdout)
if preflight.returncode != 0:
    print(preflight.stderr)
    raise RuntimeError("SUMO/TraCI preflight failed; training was not started")
for name in ("highway.net.xml", "trajectory.xml", "events.json"):
    assert (PREFLIGHT / name).is_file(), f"Missing preflight artifact: {name}"
print("SUMO/TraCI scenario preflight passed.")


def pipeline_command(*extra):
    return [
        sys.executable,
        "-B",
        str(PIPELINE),
        *extra,
        "--persistent-root",
        str(PERSISTENT_ROOT),
        "--work-root",
        str(WORK_ROOT),
        "--config",
        str(CONFIG),
    ]


def read_status():
    result = subprocess.run(
        pipeline_command("--status"), cwd=str(REPOSITORY), capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("Unable to read adaptive-demo status")
    return json.loads(result.stdout)


wall_started = time.monotonic()
for attempt in range(12):
    status = read_status()
    print(json.dumps(status, indent=2, ensure_ascii=False))
    if status["next_stage"] is None:
        break
    elapsed_minutes = (time.monotonic() - wall_started) / 60
    remaining_minutes = 295 - elapsed_minutes
    if remaining_minutes <= 35:
        raise RuntimeError(
            "The five-hour safety limit was reached after saving progress. Run this notebook again to resume."
        )
    print("Starting/resuming stage:", status["next_stage"])
    process = subprocess.Popen(
        pipeline_command("--stage", "next", "--time-budget-minutes", f"{remaining_minutes:.1f}"),
        cwd=str(REPOSITORY),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code == 75:
        print("Safe checkpoint reached; attempting automatic resume...")
        continue
    if return_code != 0:
        print("Failure diagnostics:")
        for diagnostic in sorted(PERSISTENT_ROOT.rglob("*.json")):
            if diagnostic.name in {
                "summary.json",
                "best_validation.json",
                "adaptive_demo_state.json",
            }:
                try:
                    print(diagnostic)
                    print(diagnostic.read_text(encoding="utf-8")[:8000])
                except Exception:
                    pass
        raise RuntimeError(f"Adaptive-demo stage failed with return code {return_code}")
else:
    raise RuntimeError("Automatic stage limit reached")

final_status = read_status()
assert final_status["next_stage"] is None, final_status
print(json.dumps(final_status, indent=2, ensure_ascii=False))
print("ADAPTIVE DEMO PIPELINE COMPLETED")

In [ ]:
import pandas as pd
from IPython.display import Image, display

RESULTS = PERSISTENT_ROOT / "presentation_results"
TABLE = RESULTS / "table_comparison.csv"
METRICS_FIGURE = RESULTS / "fig_metric_comparison.png"
TRAINING_FIGURE = RESULTS / "fig_training_curve.png"
SUMMARY = PERSISTENT_ROOT / "comparison_results/summary.json"

assert all(path.is_file() for path in (TABLE, METRICS_FIGURE, TRAINING_FIGURE, SUMMARY))
display(pd.read_csv(TABLE))
display(Image(filename=str(METRICS_FIGURE)))
display(Image(filename=str(TRAINING_FIGURE)))

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("Acceptance:", json.dumps(summary.get("acceptance"), indent=2))
archive = shutil.make_archive(
    "/content/G11project-directional-corridor-v1-results", "zip", PERSISTENT_ROOT
)
print("Persistent results:", RESULTS)
print("Downloadable archive:", archive)